## ML_1022: Linear Self-Attention

**Difficulty**: Hard | **TorchCode**: #12

### Problem
Implement Linear Attention using the kernel trick to reduce complexity from O(S²·d) to O(S·d²). Use the feature map φ(x) = ELU(x) + 1 and reorder the matmul.

### Formula
$$\text{out} = \frac{\phi(Q)\,(\phi(K)^T V)}{\phi(Q)\,\mathbf{1}^T \phi(K)^T}, \quad \phi(x) = \text{ELU}(x) + 1$$

In [ ]:
import torch
import torch.nn.functional as F

def linear_attention(Q, K, V):
    Q_prime = F.elu(Q) + 1
    K_prime = F.elu(K) + 1
    KV = torch.bmm(K_prime.transpose(1, 2), V)        # (B, D_k, D_v)
    Z = K_prime.sum(dim=1, keepdim=True)               # (B, 1, D_k)
    num = torch.bmm(Q_prime, KV)                       # (B, S, D_v)
    den = torch.bmm(Q_prime, Z.transpose(1, 2))        # (B, S, 1)
    return num / (den + 1e-6)

In [ ]:
import torch
import time

# --- Test 1: Output shape ---
out = linear_attention(torch.randn(2, 8, 16), torch.randn(2, 8, 16), torch.randn(2, 8, 32))
assert out.shape == (2, 8, 32)

# --- Test 2: No NaN or Inf ---
torch.manual_seed(0)
out = linear_attention(torch.randn(2, 16, 8), torch.randn(2, 16, 8), torch.randn(2, 16, 8))
assert not torch.isnan(out).any() and not torch.isinf(out).any()

# --- Test 3: Gradient flow ---
Q = torch.randn(1, 4, 8, requires_grad=True)
K = torch.randn(1, 4, 8, requires_grad=True)
V = torch.randn(1, 4, 8, requires_grad=True)
linear_attention(Q, K, V).sum().backward()
assert Q.grad is not None and K.grad is not None and V.grad is not None

# --- Test 4: Linear complexity on long sequences ---
torch.manual_seed(0)
Q = torch.randn(1, 2048, 64); K = torch.randn(1, 2048, 64); V = torch.randn(1, 2048, 64)
t0 = time.perf_counter()
for _ in range(10):
    linear_attention(Q, K, V)
assert time.perf_counter() - t0 < 5.0

print('All tests passed!')

In [ ]:
import torch
import torch.nn as nn


class LinearAttention(nn.Module):
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.elu = nn.ELU()

    def phi(self, x: torch.Tensor) -> torch.Tensor:
        return self.elu(x) + 1

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
        """
        Q: (B, S, D)
        K: (B, S, D)
        V: (B, S, Dv)

        return:
            out: (B, S, Dv)
        """
        Q_phi = self.phi(Q)              # (B, S, D)
        K_phi = self.phi(K)              # (B, S, D)

        # Compute K^T V first:
        # (B, D, S) @ (B, S, Dv) -> (B, D, Dv)
        KV = K_phi.transpose(-2, -1) @ V

        # Numerator:
        # (B, S, D) @ (B, D, Dv) -> (B, S, Dv)
        numerator = Q_phi @ KV

        # Denominator:
        # first sum over sequence positions of K_phi:
        # (B, S, D) -> (B, D)
        K_sum = K_phi.sum(dim=1)

        # (B, S, D) * (B, 1, D) -> sum over D -> (B, S, 1)
        denominator = (Q_phi * K_sum.unsqueeze(1)).sum(dim=-1, keepdim=True)

        out = numerator / (denominator + self.eps)
        return out
        
torch.manual_seed(0)

B, S, D, Dv = 2, 5, 8, 8
Q = torch.randn(B, S, D)
K = torch.randn(B, S, D)
V = torch.randn(B, S, Dv)

attn = LinearAttention()
out = attn(Q, K, V)

print(out.shape)   # torch.Size([2, 5, 8])


torch.Size([2, 5, 8])
